# RehabLLM v0.4 — Corpus Autopsy → Clean → Robotics Enrichment → Continued Pretraining → SFT

This notebook is compatible with **Google Colab and Kaggle**. It is intentionally split into two phases:

1. **CPU preparation** — audit v0.3, rebuild/persist source records, fetch targeted PMC robotics material, clean/deduplicate/mix the v0.4 corpus, and tokenise it with the **existing v0.3 tokenizer**.
2. **GPU training** — continue-pretrain from the v0.3 weights with a fresh optimiser, then run a small instruction-tuning smoke test.

> The bundled instruction seed is only a pipeline smoke test. It is not enough instruction data for a production assistant.


## 1. Settings


In [ ]:
REPO_URL = "https://github.com/ZenKOH/RehabLLM.git"
REPO_BRANCH = "main"
PERSIST_FOLDER = "RehabLLM-v04"
BASE_TARGET_DOCS = 4_000
V04_TARGET_DOCS = 4_500
ROBOTICS_RETMAX_PER_QUERY = 150
ROBOTICS_MAX_ARTICLES = 1_800

# NCBI requests a contact email for E-Utilities. Enter yours before the PMC enrichment cell.
NCBI_EMAIL = ""

# Optional manual overrides. Leave blank to use the Colab/Kaggle defaults below.
V03_CHECKPOINT_OVERRIDE = ""
V03_TOKENIZER_OVERRIDE = ""


## 2. Detect runtime and mount persistent storage


In [ ]:
import os, sys, shutil, subprocess, json
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = Path("/kaggle").exists() or "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_ROOT = Path("/content")
    PERSIST_ROOT = Path("/content/drive/MyDrive") / PERSIST_FOLDER
    DEFAULT_V03_ROOT = Path("/content/drive/MyDrive/RehabLLM-FreeGPU")
    RUNTIME = "Google Colab"
elif IN_KAGGLE:
    WORK_ROOT = Path("/kaggle/working")
    PERSIST_ROOT = WORK_ROOT / PERSIST_FOLDER
    DEFAULT_V03_ROOT = Path("/kaggle/input/rehabllm-v03")
    RUNTIME = "Kaggle"
else:
    WORK_ROOT = Path.cwd()
    PERSIST_ROOT = WORK_ROOT / PERSIST_FOLDER
    DEFAULT_V03_ROOT = WORK_ROOT / "RehabLLM-FreeGPU"
    RUNTIME = "Other/Jupyter"

PERSIST_ROOT.mkdir(parents=True, exist_ok=True)
PREP_ROOT = PERSIST_ROOT / "prep"
PREP_ROOT.mkdir(parents=True, exist_ok=True)

V03_CHECKPOINT = Path(V03_CHECKPOINT_OVERRIDE) if V03_CHECKPOINT_OVERRIDE else DEFAULT_V03_ROOT / "checkpoints/free_gpu_v0.3/final.pt"
V03_TOKENIZER = Path(V03_TOKENIZER_OVERRIDE) if V03_TOKENIZER_OVERRIDE else DEFAULT_V03_ROOT / "data_cache/processed/rehab_sp.model"

print("runtime:", RUNTIME)
print("persistent root:", PERSIST_ROOT)
print("v0.3 checkpoint:", V03_CHECKPOINT, V03_CHECKPOINT.exists())
print("v0.3 tokenizer:", V03_TOKENIZER, V03_TOKENIZER.exists())


## 3. Clone/update RehabLLM and install dependencies


In [ ]:
REPO_DIR = WORK_ROOT / "RehabLLM"
if (REPO_DIR / ".git").exists():
    subprocess.run(["git","-C",str(REPO_DIR),"fetch","origin"],check=True)
    subprocess.run(["git","-C",str(REPO_DIR),"checkout",REPO_BRANCH],check=True)
    subprocess.run(["git","-C",str(REPO_DIR),"reset","--hard",f"origin/{REPO_BRANCH}"],check=True)
else:
    if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
    subprocess.run(["git","clone","--branch",REPO_BRANCH,REPO_URL,str(REPO_DIR)],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-e",f"{REPO_DIR}[cloud]"],check=True)
os.chdir(REPO_DIR)
print("commit:", subprocess.check_output(["git","rev-parse","HEAD"],text=True).strip())


## 4. Optional v0.3 corpus autopsy

If the earlier v0.3 text cache exists, this audits the actual cached training/validation/test text for configured boilerplate and domain mix.


In [ ]:
V03_CURATED = DEFAULT_V03_ROOT / "data_cache/curated"
autopsy_inputs = [V03_CURATED / f"{split}.txt" for split in ("train","val","test")]
autopsy_inputs = [p for p in autopsy_inputs if p.exists()]
if autopsy_inputs:
    cmd = [sys.executable,"scripts/autopsy_corpus.py","--input",*[str(p) for p in autopsy_inputs],
           "--out-json",str(PREP_ROOT / "v03_autopsy.json"),
           "--out-md",str(PREP_ROOT / "v03_autopsy.md")]
    subprocess.run(cmd,check=True)
else:
    print("No v0.3 text cache found; skipping text autopsy.")


## 5. Build or restore a persistent broad rehabilitation source corpus

This is CPU/network work. On the first run it may take a long time. The resulting JSONL is saved persistently so it is not repeated on later runs.


In [ ]:
BASE_DIR = PREP_ROOT / "base_common_pile"
BASE_ARTICLES = BASE_DIR / "articles.jsonl"
if BASE_ARTICLES.exists():
    print("Restoring persistent base corpus:", BASE_ARTICLES)
else:
    BASE_DIR.mkdir(parents=True,exist_ok=True)
    subprocess.run([
        sys.executable,"scripts/build_hf_rehab_corpus.py",
        "--target-docs",str(BASE_TARGET_DOCS),
        "--min-docs","2000",
        "--max-scanned","1000000",
        "--out-dir",str(BASE_DIR),
    ],check=True)
print("base articles:", BASE_ARTICLES, BASE_ARTICLES.exists())


## 6. Build or restore targeted PMC robotics enrichment

Enter `NCBI_EMAIL` in Step 1 before running this cell. The file is persisted after the first successful retrieval.


In [ ]:
ROBOTICS_ARTICLES = PREP_ROOT / "pmc_robotics_v04.jsonl"
if ROBOTICS_ARTICLES.exists():
    print("Restoring persistent robotics enrichment:", ROBOTICS_ARTICLES)
else:
    if not NCBI_EMAIL.strip():
        raise ValueError("Set NCBI_EMAIL in Step 1, then rerun this cell.")
    os.environ["NCBI_EMAIL"] = NCBI_EMAIL.strip()
    subprocess.run([
        sys.executable,"scripts/fetch_pmc.py",
        "--profile","robotics",
        "--retmax-per-query",str(ROBOTICS_RETMAX_PER_QUERY),
        "--max-articles",str(ROBOTICS_MAX_ARTICLES),
        "--out",str(ROBOTICS_ARTICLES),
    ],check=True)
print("robotics enrichment:", ROBOTICS_ARTICLES, ROBOTICS_ARTICLES.exists())


## 7. Clean, deduplicate and build the v0.4 mixture


In [ ]:
V04_DATA = PREP_ROOT / "v04_data"
V04_STATS = V04_DATA / "v04_corpus_stats.json"
if V04_STATS.exists():
    print("Restoring prepared v0.4 corpus:", V04_DATA)
else:
    subprocess.run([
        sys.executable,"scripts/build_v04_corpus.py",
        "--input",str(BASE_ARTICLES),str(ROBOTICS_ARTICLES),
        "--out-dir",str(V04_DATA),
        "--target-docs",str(V04_TARGET_DOCS),
    ],check=True)
print(json.dumps(json.loads(V04_STATS.read_text()),indent=2)[:12000])


## 8. Tokenise v0.4 with the existing v0.3 tokenizer

**Do not train a new tokenizer.** The existing embedding/output rows depend on the original 8,000-token vocabulary.


In [ ]:
if not V03_TOKENIZER.exists():
    raise FileNotFoundError(f"v0.3 tokenizer not found: {V03_TOKENIZER}")
PROCESSED = PREP_ROOT / "v04_processed"
required = [PROCESSED / f"{s}.bin" for s in ("train","val","test")]
if all(p.exists() for p in required):
    print("Restoring tokenised v0.4 corpus:", PROCESSED)
else:
    PROCESSED.mkdir(parents=True,exist_ok=True)
    subprocess.run([
        sys.executable,"scripts/prepare_data.py",
        "--train-text",str(V04_DATA / "train.txt"),
        "--val-text",str(V04_DATA / "val.txt"),
        "--test-text",str(V04_DATA / "test.txt"),
        "--tokenizer",str(V03_TOKENIZER),
        "--out-dir",str(PROCESSED),
    ],check=True)
import numpy as np
for split in ("train","val","test"):
    p=PROCESSED/f"{split}.bin"
    print(split, f"{p.stat().st_size // np.dtype(np.int32).itemsize:,} tokens")


## 9. Prepare the instruction seed smoke-test set


In [ ]:
INSTRUCTION_DIR = PREP_ROOT / "instructions_seed"
if not (INSTRUCTION_DIR / "train.jsonl").exists():
    subprocess.run([
        sys.executable,"scripts/prepare_instructions.py",
        "--input","data/instruction_seed.jsonl",
        "--out-dir",str(INSTRUCTION_DIR),
    ],check=True)
print((INSTRUCTION_DIR / "stats.json").read_text())


## CPU preparation complete

At this point the expensive retrieval/cleaning work is safely stored. If you are on a CPU runtime, switch to a GPU now and rerun the notebook from the top. The preparation cells will restore their cached outputs instead of rebuilding them.


## 10. GPU readiness gate

This checks whether the prepared corpus is worth spending GPU time on. The thresholds are intentionally much stronger than the v0.3 mix: at least 3,500 selected documents, 12% robotics, 5% neurotechnology, 20% robotics+neurotechnology combined, at least 100 validation/test documents each, and no evidence that cleaning removed an implausibly large fraction of the source text.


In [ ]:
READINESS_PATH = PREP_ROOT / "v04_readiness.json"
result = subprocess.run([
    sys.executable, "scripts/assess_v04_readiness.py",
    "--stats", str(V04_STATS),
    "--out", str(READINESS_PATH),
], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
READINESS = json.loads(READINESS_PATH.read_text())
READINESS_READY = bool(READINESS["ready"])
print("\nREADY FOR GPU:", READINESS_READY)


## 11. Verify GPU before training


In [ ]:
if not READINESS_READY:
    raise RuntimeError("Corpus readiness gate says HOLD. Improve/rebuild the corpus before enabling GPU training.")
import torch
print("PyTorch:",torch.__version__)
print("CUDA available:",torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Preparation is complete. Enable a GPU runtime, then rerun the notebook from the top.")
print("GPU:",torch.cuda.get_device_name(0))


## 12. Restore prepared token files locally for faster GPU access


In [ ]:
LOCAL_PROCESSED = REPO_DIR / "data/v04/processed"
LOCAL_PROCESSED.mkdir(parents=True,exist_ok=True)
for name in ("train.bin","val.bin","test.bin"):
    shutil.copy2(PROCESSED/name, LOCAL_PROCESSED/name)
print("local processed:", LOCAL_PROCESSED)


## 13. Continue pretraining from v0.3 weights


In [ ]:
if not V03_CHECKPOINT.exists():
    raise FileNotFoundError(f"v0.3 checkpoint not found: {V03_CHECKPOINT}")
DOMAIN_OUT = PERSIST_ROOT / "checkpoints/v04_domain"
DOMAIN_OUT.mkdir(parents=True,exist_ok=True)
DOMAIN_FINAL = DOMAIN_OUT / "final.pt"
if DOMAIN_FINAL.exists():
    print("Domain continuation already complete:",DOMAIN_FINAL)
else:
    partial = sorted(DOMAIN_OUT.glob("step_*.pt"))
    cmd=[sys.executable,"scripts/train_model.py","--config","configs/v04_domain.yaml",
         "--train",str(LOCAL_PROCESSED/"train.bin"),"--val",str(LOCAL_PROCESSED/"val.bin"),
         "--out",str(DOMAIN_OUT)]
    if partial:
        cmd += ["--resume","latest"]
    else:
        cmd += ["--init-checkpoint",str(V03_CHECKPOINT)]
    subprocess.run(cmd,check=True)
print("domain final:",DOMAIN_FINAL.exists())


## 14. Instruction-tuning smoke test

This uses only the bundled seed examples and the deliberately short seed schedule. It validates the SFT mechanism; it does **not** establish production-quality instruction following.


In [ ]:
SFT_OUT = PERSIST_ROOT / "checkpoints/v04_sft_seed"
SFT_OUT.mkdir(parents=True,exist_ok=True)
SFT_FINAL = SFT_OUT / "final.pt"
if SFT_FINAL.exists():
    print("Seed SFT already complete:",SFT_FINAL)
else:
    cmd=[sys.executable,"scripts/train_sft.py","--config","configs/v04_sft_seed.yaml",
         "--base-checkpoint",str(DOMAIN_FINAL),"--tokenizer",str(V03_TOKENIZER),
         "--train",str(INSTRUCTION_DIR/"train.jsonl"),"--val",str(INSTRUCTION_DIR/"val.jsonl"),
         "--out",str(SFT_OUT)]
    if list(SFT_OUT.glob("step_*.pt")):
        cmd += ["--resume","latest"]
    subprocess.run(cmd,check=True)
print("sft final:",SFT_FINAL.exists())


## 15. Direct-answer behavioural test


In [ ]:
prompt = "Explain robot-assisted rehabilitation after stroke, including important evidence limitations."
result=subprocess.run([
    sys.executable,"scripts/chat_sft.py",
    "--checkpoint",str(SFT_FINAL),
    "--tokenizer",str(V03_TOKENIZER),
    "--instruction",prompt,
    "--max-new-tokens","160",
],capture_output=True,text=True)
print("return code:",result.returncode)
print("\n--- RehabLLM v0.4 seed-SFT answer ---\n")
print(result.stdout)
if result.stderr: print("\n--- diagnostic ---\n",result.stderr)


## 16. Package v0.4 artefacts


In [ ]:
PACKAGE = PERSIST_ROOT / "package"
if PACKAGE.exists(): shutil.rmtree(PACKAGE)
PACKAGE.mkdir(parents=True)
for src in [DOMAIN_FINAL,SFT_FINAL,PREP_ROOT/"v03_autopsy.json",V04_STATS,READINESS_PATH,INSTRUCTION_DIR/"stats.json"]:
    if src.exists():
        name = "domain_final.pt" if src == DOMAIN_FINAL else "sft_seed_final.pt" if src == SFT_FINAL else src.name
        shutil.copy2(src,PACKAGE/name)
shutil.copy2(V03_TOKENIZER,PACKAGE/"rehab_sp.model")
archive=shutil.make_archive(str(PERSIST_ROOT/"RehabLLM_v04_seed_pipeline"),"zip",root_dir=PACKAGE)
print("archive:",archive)
